# 11 Knowledge graph facts from the machine manual

**Goal:** read `machine_manual.pdf` (66 pages) automatically and turn it into the facts the knowledge graph needs:
which causes lead to which defect (and in what order), the fix for each cause, which alarm code points to which cause,
maintenance intervals, operating rules and the cutting parameters. Notebook 04 types these in by hand; this notebook
gets them from the PDF, the way you would have to with a real 1000-page manual.

**The method** (sources in brackets, details in the methodology doc):
1. **Parse** the PDF with `pdfplumber`, keeping text and tables apart and dropping repeated headers/footers
   (pdfplumber was best on tables in El Bachyr et al. 2026).
2. **Split by section**, using the heading font size, so every fact keeps its section and page (provenance).
3. **Follow cross-references**: "see section 8.3" gets the text of 8.3 appended.
4. **Parse regular tables with code** (cutting tables, message list): exact, no LLM needed.
5. **LLM extraction for the prose and small tables**, guided by a fixed schema (Text2KGBench; EDC "extract" step)
   with one worked example (in-context learning, van Cauter & Yakovets 2024), answered as JSON that must match a schema.
6. **Checks**: allowed relation and types, the quoted evidence must really be in the text (grounding),
   and chapters that say they do not apply are dropped (scope).
7. **Canonicalise**: different wordings of the same cause are merged into one id (EDC "canonicalise"; KGGen clustering).
8. **Score** against the gold list: recall per fact type and per hard case, plus how many traps were extracted.

**Needs:** `pip install pdfplumber` once; Ollama running with `llama3.1:8b` for step 5 (as in notebook 05).
To score, copy `manual_gold_facts.csv` into the `data/` folder (it is not in the repo, like an answer key).
**Outputs:** `data/processed/manual_facts.csv` (all facts with section and page) and the four tables notebook 04 uses.


In [ ]:
from pathlib import Path
import json, re, time, difflib
from collections import Counter
import pandas as pd
import pdfplumber
import requests

DATA = Path("..") / "datasets"
PDF = DATA / "documents" / "machine_manual.pdf"
OUT = Path("..") / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
GOLD = Path("..") / "data" / "manual_gold_facts.csv"     # the scoring list (not in the repo); copy it here to score

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.1:8b"
try:
    have = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    OLLAMA_OK = any(h.startswith(MODEL) for h in have)
    print("Ollama is running. Models:", have)
except requests.exceptions.RequestException:
    OLLAMA_OK = False
    print("Ollama is not running: steps 1-4 still run, the LLM step (5) is skipped.")


## 1. Parse the PDF
Each page becomes a list of **blocks** in reading order: one per line of text (with its font size) and one per table.
A table is turned into rows like `| 1 | Focus position wrong | Run focus test |`, which LLMs read well.
Lines that repeat on more than half of the pages (the footer) are dropped.


In [ ]:
def table_md(rows):
    rows = [[(c or "").replace("\n", " ").strip() for c in r] for r in rows]
    return "\n".join("| " + " | ".join(r) + " |" for r in rows)

pages = []
with pdfplumber.open(PDF) as pdf:
    for pno, page in enumerate(pdf.pages, 1):
        tables = page.find_tables()
        boxes = [t.bbox for t in tables]
        inside = lambda l: any(b[0] - 1 <= l["x0"] and l["top"] >= b[1] - 1 and l["bottom"] <= b[3] + 1 for b in boxes)
        blocks = [{"top": l["top"], "kind": "text", "size": round(max(c["size"] for c in l["chars"]), 1), "text": l["text"]}
                  for l in page.extract_text_lines() if not inside(l)]
        blocks += [{"top": t.bbox[1], "kind": "table", "size": 0, "text": table_md(t.extract()), "rows": t.extract()} for t in tables]
        for b in sorted(blocks, key=lambda b: b["top"]):
            b["page"] = pno
            pages.append(b)

# lines that repeat on most pages are headers/footers: drop them (page numbers removed first)
norm = lambda s: re.sub(r"\d+", "#", s)
rep = Counter(norm(b["text"]) for b in pages if b["kind"] == "text")
n_pages = pages[-1]["page"]
blocks = [b for b in pages if not (b["kind"] == "text" and rep[norm(b["text"])] > n_pages / 2)]
print(f"{n_pages} pages -> {len(blocks)} blocks ({sum(b['kind'] == 'table' for b in blocks)} tables)")


## 2. Sections and chunks
A line in a big font that starts with a number (`10.1 Dross ...`) or a letter (`C Cell 1 ...`) starts a new **section**.
Chunking by section, not by a fixed number of characters, keeps a table together with its heading,
and gives every fact a section number to cite. Sections longer than about 3000 characters (the message list)
are cut between table rows, repeating the header.


In [ ]:
HEAD = re.compile(r"^((?:\d{1,2}|[A-Z])(?:\.\d{1,2})?)\s+\S")
sections, cur = [], {"id": "front", "title": "Front matter", "page": 1, "blocks": []}
for b in blocks:
    m = HEAD.match(b["text"]) if b["kind"] == "text" else None
    if b["kind"] == "text" and b["size"] >= 12 and m:
        sections.append(cur)
        cur = {"id": m.group(1), "title": b["text"], "page": b["page"], "blocks": []}
    else:
        cur["blocks"].append(b)
sections.append(cur)
for s in sections:
    s["chapter"] = s["id"].split(".")[0]
    s["text"] = "\n".join(b["text"] for b in s["blocks"])
sec_by_id = {s["id"]: s for s in sections}
chapter_title = {s["id"]: s["title"] for s in sections if "." not in s["id"]}
print(len(sections), "sections, e.g.", [s["title"] for s in sections[40:44]])

MAX_CHARS = 3000
def chunks_of(s):
    """One chunk per section; long sections are cut between blocks (long tables between rows, header repeated)."""
    out, buf = [], ""
    for b in s["blocks"]:
        parts = [b["text"]]
        if b["kind"] == "table" and len(b["text"]) > MAX_CHARS:
            head, body = b["text"].split("\n", 1)[0], b["text"].split("\n")[1:]
            step = max(1, MAX_CHARS // max(len(r) for r in body))
            parts = [head + "\n" + "\n".join(body[i:i + step]) for i in range(0, len(body), step)]
        for p in parts:
            if buf and len(buf) + len(p) > MAX_CHARS:
                out.append(buf); buf = ""
            buf += p + "\n"
    if buf.strip():
        out.append(buf)
    return out

chunks = []
for s in sections:
    for k, text in enumerate(chunks_of(s)):
        chunks.append({"chunk_id": f"{s['id']}#{k}", "section": s["id"], "chapter": s["chapter"],
                       "page": s["page"], "title": s["title"], "text": text})
print(len(chunks), "chunks; longest", max(len(c["text"]) for c in chunks), "characters")


## 3. Cross-references
Manuals often say *"see section 8.3"* instead of repeating the fix. Without the referenced text the LLM cannot know
what message 2105 points to. So the referenced section (first 1200 characters) is appended to the chunk.


In [ ]:
REF = re.compile(r"\b(?:see\s+)?section\s+(\d+\.\d+)", re.I)
def with_refs(c):
    refs = [r for r in dict.fromkeys(REF.findall(c["text"])) if r in sec_by_id and r != c["section"]]
    extra = "".join(f"\n[Referenced section {r} {sec_by_id[r]['title']}]\n{sec_by_id[r]['text'][:1200]}\n" for r in refs)
    return c["text"] + extra, refs
for c in chunks:
    c["text_ext"], c["refs"] = with_refs(c)
print(sum(bool(c["refs"]) for c in chunks), "chunks got a referenced section appended")


## 4. Regular tables: parsed by code
The cutting tables and the message list have the same columns on every page, so simple code reads them exactly.
Using an LLM here would only add mistakes and time (a choice to defend in the report: *rules where the format is fixed,
LLM where it is free text*). An action like "See section 8.3." gets the referenced text attached, as in step 3.


In [ ]:
rule_facts = []
def add_rule(rel, subj, obj, s, page, qual=""):
    rule_facts.append({"relation": rel, "subject": subj, "object": obj, "rank": None, "gas": None, "qualifier": qual,
                       "evidence": "table row", "section": s["id"], "page": page, "source": "rule"})

PARAM_COLS = {"Power (W)": "laser_power_W", "Feed (mm/min)": "feed_rate_mm_min", "Pressure (bar)": "gas_pressure_bar",
              "Focus (mm)": "focus_mm", "Nozzle (mm)": "nozzle_mm"}
for s in sections:
    for b in s["blocks"]:
        if b["kind"] != "table":
            continue
        head = [re.sub(r"\s+", " ", h or "").strip() for h in b["rows"][0]]
        if head[:1] == ["Thickness (mm)"]:                              # cutting table
            m = re.search(r"\b(S235JR|1\.4301|AlMg3)\b.*\((O2|N2)\)", s["title"])
            for r in b["rows"][1:]:
                for h, v in zip(head[1:], r[1:]):
                    add_rule("PARAM", f"{m.group(1)}|{m.group(2)}|{r[0]}", float(v), s, b["page"], PARAM_COLS[h])
        if head == ["Code", "Level", "Text", "Action"]:                 # message list
            for code, lvl, text, act in b["rows"][1:]:
                add_rule("ALARM_TEXT", code, text.replace("\n", " "), s, b["page"])
                add_rule("ALARM_LEVEL", code, lvl, s, b["page"])
                act = act.replace("\n", " ")
                for r in REF.findall(act):                              # resolve "See section 8.3."
                    act += f" [{r}: {sec_by_id[r]['text'][:600]}]" if r in sec_by_id else ""
                add_rule("ALARM_ACTION", code, act, s, b["page"])
rule_facts = pd.DataFrame(rule_facts)
print(rule_facts["relation"].value_counts().to_dict())


## 5. LLM extraction (prose and small tables)
Every remaining chunk that mentions a maintenance or quality word goes to the LLM with:
- the **schema**: the five relations it may use and what subject and object mean (the same edges as notebook 04's `SCHEMA`);
- the **defect names** from the inspection records, so `CAUSES` points at defects the event data uses;
- one **worked example** (a made-up section and its answer);
- a **JSON schema** passed to Ollama (`format`), so the answer always has the right fields.

The model also copies a short **evidence** quote and says whether the fact **applies** (not withdrawn, not another machine).
Answers are cached in `manual_llm_raw_main.jsonl`, so if it stops, re-running continues where it left off.
Set `LIMIT = 5` for a quick test. With `llama3.1:8b` on a laptop, expect roughly 10-30 minutes for all chunks.


In [ ]:
DEFECTS = ["dross", "rough_cut", "incomplete_cut", "burnt_edge", "bevel_deviation"]   # the defect names in the inspection records
RELATIONS = ["CAUSES", "FIXED_BY", "INDICATES", "INTERVAL", "RULE"]
SCHEMA = {"type": "object", "properties": {"facts": {"type": "array", "items": {"type": "object", "properties": {
    "relation": {"type": "string", "enum": RELATIONS},
    "subject": {"type": "string"}, "object": {"type": "string"},
    "rank": {"type": ["integer", "null"]}, "gas": {"type": ["string", "null"], "enum": ["N2", "O2", None]},
    "applies": {"type": "boolean"}, "evidence": {"type": "string"}},
    "required": ["relation", "subject", "object", "rank", "gas", "applies", "evidence"]}}}, "required": ["facts"]}

SYSTEM = f"""You extract facts from one section of a laser cutting machine manual into a knowledge graph.
Only use these relations:
- CAUSES: subject = a cause of a cut-quality defect, written as in the text; object = the defect, one of {DEFECTS};
  rank = its position in the list (1 = most likely) if the text gives an order; gas = "N2" or "O2" if the text limits it to that gas.
- FIXED_BY: subject = a cause; object = the remedy for it.
- INDICATES: subject = a 4-digit machine message code; object = the cause that this message points to.
- INTERVAL: subject = a maintenance item; object = how often to check, clean or replace it.
- RULE: subject = a short name; object = an operating rule (a limit or a prohibition) as one sentence.
evidence = a short quote copied word for word from the section that states the fact.
applies = false if the text says the value is withdrawn, from an old revision, or for a different machine; otherwise true.
Only extract facts that the section states. If it states none, return {{"facts": []}}. Answer in JSON."""

EXAMPLE_IN = "Section 9.9 Example\n| Rank | Cause | Remedy |\n| 1 | Lens dirty | Clean lens |\nMessage 1234 means the lens is dirty."
EXAMPLE_OUT = json.dumps({"facts": [
    {"relation": "CAUSES", "subject": "Lens dirty", "object": "incomplete_cut", "rank": 1, "gas": None, "applies": True, "evidence": "1 | Lens dirty"},
    {"relation": "FIXED_BY", "subject": "Lens dirty", "object": "Clean lens", "rank": None, "gas": None, "applies": True, "evidence": "Lens dirty | Clean lens"},
    {"relation": "INDICATES", "subject": "1234", "object": "Lens dirty", "rank": None, "gas": None, "applies": True, "evidence": "Message 1234 means the lens is dirty"}]})
# the example's heading names no defect, so it says incomplete_cut only to show the format

KEYWORDS = re.compile(r"cause|remed|defect|dross|rough|incomplete|burn|bevel|message|interval|replace|clean|calibrat|"
                      r"nozzle|glass|gas|feed|program|NOK|every|weekly|daily|shift", re.I)

def ask(prompt):
    try:
        resp = requests.post(f"{OLLAMA_URL}/api/chat", timeout=900, json={
            "model": MODEL, "stream": False, "format": SCHEMA, "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 3000},  # cap the answer so a model stuck repeating facts stops; long fact lists still fit
            "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": EXAMPLE_IN},
                         {"role": "assistant", "content": EXAMPLE_OUT}, {"role": "user", "content": prompt}]})
    except requests.exceptions.Timeout:
        print("  timed out, skipped (re-run the cell later to retry it)")
        return "timeout"
    resp.raise_for_status()
    try:
        return json.loads(resp.json()["message"]["content"]).get("facts", [])
    except json.JSONDecodeError:
        cut = resp.json().get("done_reason") == "length"
        print("  answer cut off at num_predict, facts lost" if cut else "  answer was not valid JSON")
        return None

def ask_split(head, text, depth=2):
    """Ask once; if the answer is cut off or not JSON, ask again on each half of the section (twice at most)."""
    facts = ask(head + text)
    lines = text.strip().split("\n")
    if facts is None and depth and len(lines) > 1:
        print(f"  retrying in two halves")
        mid = len(lines) // 2
        parts = [ask_split(head, "\n".join(h) + "\n", depth - 1) for h in (lines[:mid], lines[mid:])]
        if "timeout" in parts:
            return "timeout"
        if any(p is not None for p in parts):
            return [f for p in parts for f in (p or [])]
    return facts

def extract(use_refs=True, limit=None, tag="main", only=None):
    path = OUT / f"manual_llm_raw_{tag}.jsonl"                           # cache: re-running skips chunks already done
    # done = has facts, or already failed after the split retry; older failed answers are tried again once
    lines = [json.loads(l) for l in open(path, encoding="utf-8")] if path.exists() else []
    done = {r["chunk_id"] for r in lines if r["facts"] is not None or r.get("split_tried")}
    todo = [c for c in chunks if KEYWORDS.search(c["text"]) and c["chunk_id"] not in done
            and not c["text"].lstrip().startswith("| Code | Level") and (only is None or c["chunk_id"] in only)][:limit]
    print(f"{tag}: {len(todo)} chunks to send ({len(done)} cached)")
    t0 = time.time()
    skipped, bad = [], []
    with open(path, "a", encoding="utf-8") as f:
        for i, c in enumerate(todo, 1):
            text = c["text_ext"] if use_refs else c["text"]
            facts = ask_split(f"Chapter: {chapter_title.get(c['chapter'], '')}\nSection {c['title']}\n", text)
            if facts == "timeout":                                       # not cached, so the next run tries it again
                skipped.append(c["chunk_id"])
                continue
            if facts is None:                                            # cached as no facts (same answer at temperature 0)
                bad.append(c["chunk_id"])
            f.write(json.dumps({"chunk_id": c["chunk_id"], "facts": facts, "split_tried": facts is None}) + "\n")
            f.flush()
            if i % 20 == 0:
                print(f"  {i}/{len(todo)} chunks, {time.time() - t0:.0f} s")
    if skipped:
        print(f"{len(skipped)} chunks timed out and were skipped; re-run this cell to retry them: {skipped}")
    if bad:
        print(f"{len(bad)} chunks gave no usable JSON and count as no facts: {bad}")
    return path

def load_raw(path):
    rows = []
    by_id = {c["chunk_id"]: c for c in chunks}
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        c = by_id[r["chunk_id"]]
        for f in r["facts"] or []:
            rows.append({**f, "chunk_id": c["chunk_id"], "section": c["section"], "page": c["page"], "source": "llm",
                         "chunk_text": c["text_ext"]})
    return pd.DataFrame(rows)


In [ ]:
LIMIT = None          # e.g. 5 for a quick test
if OLLAMA_OK:
    raw_path = extract(limit=LIMIT)
    raw = load_raw(raw_path)
    print(len(raw), "facts proposed by the LLM")
    display(raw[["relation", "subject", "object", "rank", "gas", "applies", "section"]].head(15))


## 6. Checks
Three filters, each one a known LLM failure:
- **type check**: relation is one of the five, `INDICATES` starts at a 4-digit code, `CAUSES` ends at a known defect;
- **grounding**: at least 80 % of the words in the evidence quote must be in the chunk, otherwise the fact is treated as made up;
- **scope**: the model's own `applies` flag, plus a chapter-level rule: a chapter that says *does not apply / withdrawn / legacy*
  is dropped as a whole (Appendix C here).


In [ ]:
def tokens(s):
    return set(re.findall(r"[a-z0-9]+(?:[.,-][0-9]+)*", str(s).lower()))

def grounded(evidence, text):
    """Most of the quoted words must really be in the chunk: catches made-up facts."""
    ev = tokens(evidence)
    return bool(ev) and len(ev & tokens(text)) / len(ev) >= 0.8

def chapter_scope():
    """Chapter-level scope: one rule on the chapter's own words, so a whole appendix for another machine is dropped."""
    out = {}
    for s in sections:
        if "." in s["id"] or s["id"] == "front":
            continue
        txt = " ".join(x["text"] for x in sections if x["chapter"] == s["id"])[:3000].lower()
        out[s["id"]] = not re.search(r"does not apply|not valid for|withdrawn|legacy", txt)
    return out
SCOPE = chapter_scope()
print("chapters marked as not applying:", [k for k, v in SCOPE.items() if not v])

def check(df, use_grounding=True, use_scope=True):
    df = df.copy()
    ok = df["relation"].isin(RELATIONS)
    ok &= ~((df["relation"] == "INDICATES") & ~df["subject"].astype(str).str.fullmatch(r"\d{4}"))
    ok &= ~((df["relation"] == "CAUSES") & ~df["object"].isin(DEFECTS))
    df["valid_type"] = ok
    df["grounded"] = [grounded(e, t) for e, t in zip(df["evidence"], df["chunk_text"])]
    df["in_scope"] = df["applies"].fillna(True).astype(bool) & df["section"].map(lambda s: SCOPE.get(s.split(".")[0], True))
    keep = df["valid_type"] & (df["grounded"] | (not use_grounding)) & (df["in_scope"] | (not use_scope))
    return df[keep].drop(columns=["chunk_text"]), df


In [ ]:
if OLLAMA_OK:
    facts, checked = check(raw)
    print(f"kept {len(facts)} of {len(checked)}: dropped {(~checked.valid_type).sum()} wrong type, "
          f"{(~checked.grounded).sum()} not grounded, {(~checked.in_scope).sum()} out of scope")


## 7. Canonicalise causes
The same cause shows up as *"Protective glass contaminated or damaged (beam power reduced)"*, *"dirty optics"*,
*"protective glass contaminated"*... Wordings are merged when they are very similar (difflib ratio >= 0.8)
or when one's words are all inside the other's. Each group gets one id. This is the step that turns text into graph nodes.
Words in brackets are ignored. Check the groups by eye: this is where an 8B model's wording drifts most.

**Then a cause filter:** a cause node is kept only if the troubleshooting chapter lists it for a defect, which is the `CAUSES` rule
of notebook 04's schema (Cause → Defect). The LLM also calls gases (O2), defect names and machine noises "causes"; this drops them.
Facts that point to a dropped cause are dropped with it.

In [ ]:
def clean(s):
    return re.sub(r"\s+", " ", re.sub(r"\(.*?\)", "", str(s).lower())).strip(" .;:")

def canonicalise(facts, thr=0.8):
    """Cluster cause wordings (CAUSES/FIXED_BY subject, INDICATES object); each cluster gets one id."""
    names = list(facts.loc[facts["relation"].isin(["CAUSES", "FIXED_BY"]), "subject"]) + \
            list(facts.loc[facts["relation"] == "INDICATES", "object"])
    counts = Counter(clean(n) for n in names if clean(n))
    clusters = []                                                   # [canonical, members]
    for name, _ in counts.most_common():
        for cl in clusters:
            if difflib.SequenceMatcher(None, name, cl[0]).ratio() >= thr or tokens(name) <= tokens(cl[0]):
                cl[1].append(name); break
        else:
            clusters.append([name, [name]])
    cid = {}
    for canon, members in clusters:
        slug = "_".join(re.findall(r"[a-z0-9]+", canon)[:4])
        for m in members:
            cid[m] = slug
    facts = facts.copy()
    is_cause_subj = facts["relation"].isin(["CAUSES", "FIXED_BY"])
    facts.loc[is_cause_subj, "subject_id"] = facts.loc[is_cause_subj, "subject"].map(lambda s: cid.get(clean(s)))
    is_ind = facts["relation"] == "INDICATES"
    facts.loc[is_ind, "object_id"] = facts.loc[is_ind, "object"].map(lambda s: cid.get(clean(s)))
    return facts, {slug: canon for canon, m in clusters for slug in [cid[canon]]}

def keep_causes(facts, cause_names):
    """A cause must be listed for a defect in the troubleshooting chapter (the CAUSES rule of notebook 04's schema).
    This drops gases (O2), defect names and machine noises that the LLM also called causes."""
    ts = {k for k, t in chapter_title.items() if re.search(r"troubleshoot", t, re.I)}
    in_ts = facts["section"].astype(str).str.split(".").str[0].isin(ts)
    ok = set(facts.loc[(facts["relation"] == "CAUSES") & in_ts, "subject_id"].dropna())
    drop = (facts["relation"].isin(["CAUSES", "FIXED_BY"]) & ~facts["subject_id"].isin(ok)) | \
           ((facts["relation"] == "INDICATES") & ~facts["object_id"].isin(ok))
    return facts[~drop], {k: v for k, v in cause_names.items() if k in ok}


In [ ]:
if OLLAMA_OK:
    facts, cause_names = canonicalise(facts)
    n_before = len(cause_names)
    facts, cause_names = keep_causes(facts, cause_names)
    print(f"cause filter: {n_before} cause names -> {len(cause_names)}")
    display(pd.Series(cause_names, name="cause wording").to_frame())


## 8. Save in notebook 04's format
`manual_facts.csv` has every fact with its section and page. The four small tables match the dicts typed in notebook 04
(`CAUSES`, `CAUSES_DEFECT`, `INDICATES`, `ALARM_TEXT`), so notebook 04 can later load these files instead.


In [ ]:
if OLLAMA_OK:
    allf = pd.concat([facts, rule_facts], ignore_index=True)
    allf.to_csv(OUT / "manual_facts.csv", index=False)
    fix = facts[facts.relation == "FIXED_BY"].groupby("subject_id")["object"].first()
    pd.DataFrame({"cause": list(cause_names), "text": list(cause_names.values())}).assign(
        fix=lambda d: d["cause"].map(fix)).to_csv(OUT / "manual_kg_causes.csv", index=False)
    cd = facts[facts.relation == "CAUSES"].sort_values("rank").drop_duplicates(["object", "subject_id", "gas"])
    cd[["object", "subject_id", "rank", "gas"]].rename(columns={"object": "defect", "subject_id": "cause"}).to_csv(
        OUT / "manual_kg_causes_defect.csv", index=False)
    facts[facts.relation == "INDICATES"].drop_duplicates(["subject", "object_id"])[["subject", "object_id"]].rename(
        columns={"subject": "code", "object_id": "cause"}).to_csv(OUT / "manual_kg_indicates.csv", index=False)
    rule_facts[rule_facts.relation == "ALARM_TEXT"][["subject", "object"]].rename(
        columns={"subject": "code", "object": "text"}).to_csv(OUT / "manual_kg_alarm_text.csv", index=False)
    print("saved manual_facts.csv and manual_kg_*.csv to", OUT)


## 9. Score against the gold list
For each of the 147 true facts: was it found? Causes are matched by **wording**, so it does not matter what id step 7 chose.
Numbers must match exactly. Also counted: how many of the 6 **traps** (Rev 2 values, Cell 1 appendix) ended up in the graph.

**Precision** is not computed against the gold list: the gold list holds the diagnostic facts only, and the message list alone
has 150 other true codes. Instead, look at the unmatched LLM facts below by hand and count how many are wrong
(Text2KGBench calls these hallucinations).


In [ ]:
def overlap(gold, got):
    """Share of the gold wording found in the extracted wording; every number in the gold must be there too."""
    g, t = tokens(gold), tokens(got)
    if not g or any(x not in t for x in g if x[0].isdigit()):
        return 0.0
    return len(g & t) / len(g)

def score(facts, rules, gold, cause_names):
    """Each gold fact is found or not. Causes are matched by wording, so the pipeline's own ids don't matter."""
    gtext = gold[(gold["kind"] == "cause") & (gold["relation"] == "TEXT")].set_index("subject")["object"].map(clean)
    align = {slug: max(gtext.index, key=lambda g: overlap(gtext[g], name)) for slug, name in cause_names.items()
             if max(overlap(gtext[g], name) for g in gtext.index) >= 0.6}
    f = facts.copy()
    f["subj_gold"] = f.get("subject_id", pd.Series(dtype=object)).map(align)
    f["obj_gold"] = f.get("object_id", pd.Series(dtype=object)).map(align)
    rr = rules

    def found(g):
        rel, subj, obj, q = g["relation"], str(g["subject"]), str(g["object"]), str(g["qualifier"])
        if g["kind"] == "param":
            hit = rr[(rr.relation == "PARAM") & (rr.subject == subj) & (rr.qualifier == rel)]
            return bool(len(hit)) and abs(hit["object"].iloc[0] - float(obj)) < 1e-9
        if g["kind"] == "alarm" and rel in ("TEXT", "LEVEL"):
            hit = rr[(rr.relation == f"ALARM_{rel}") & (rr.subject == subj)]
            return any(overlap(obj, o) >= 0.9 for o in hit["object"])
        if rel == "ACTION":
            hit = rr[(rr.relation == "ALARM_ACTION") & (rr.subject == subj)]
            want = re.search(r"\((.*)\)", obj).group(1)
            return any(overlap(want, o) >= 0.9 for o in hit["object"])
        if rel == "INDICATES":
            return bool(len(f[(f.relation == "INDICATES") & (f.subject.astype(str) == subj) & (f.obj_gold == obj)]))
        if rel == "CAUSES":
            kw = dict(x.split("=") for x in q.split(";")) if "=" in q else {}
            hit = f[(f.relation == "CAUSES") & (f.subj_gold == subj) & (f.object == obj)]
            g_ok = hit if "gas" not in kw else hit[hit.gas == kw["gas"]]
            return bool(len(g_ok[g_ok["rank"] == int(kw["rank"])]))
        if rel == "TEXT" and g["kind"] == "cause":
            return subj in align.values()
        if rel == "FIXED_BY":
            hit = f[(f.relation == "FIXED_BY") & (f.subj_gold == subj)]
            return any(overlap(obj, o) >= 0.6 for o in hit["object"])
        if rel == "INTERVAL":
            hit = f[(f.relation == "INTERVAL")]
            return any(overlap(subj, s) >= 0.6 and overlap(obj, o) >= 0.6 for s, o in zip(hit.subject, hit.object))
        return any(overlap(obj, o) >= 0.7 for o in f["object"])          # RULE, SPARES: by wording

    def trap(g):
        """A trap counts as extracted if the wrong value made it into the graph."""
        subj, rel, obj = str(g["subject"]), g["relation"], str(g["object"])
        from_trap = f["section"].astype(str).str.split(".").str[0] == str(g["section"]).split(".")[0]
        if rel.startswith("CAUSES"):     # the other machine's ranking: rank 1 for focus, taken from that section
            return bool(len(f[from_trap & (f.relation == "CAUSES") & (f.subj_gold == obj) & (f.object == subj) & (f["rank"] == 1)]))
        if rel == "INDICATES":           # the other machine's meaning of the code
            return bool(len(f[from_trap & (f.relation == "INDICATES") & (f.subject.astype(str) == subj)]))
        return any(overlap(obj, o) >= 0.7 for o in f["object"])     # old or other-machine values: by wording and numbers

    valid, traps = gold[gold["valid"] == "yes"].copy(), gold[gold["valid"] == "no"].copy()
    valid["found"] = valid.apply(found, axis=1)
    traps["extracted"] = traps.apply(trap, axis=1)
    return valid, traps, align


In [ ]:
HAVE_GOLD = GOLD.exists()
if not HAVE_GOLD:
    print("No gold list at", GOLD, "- copy manual_gold_facts.csv there to score.")

def report(name, facts, cause_names):
    valid, traps, _ = score(facts, rule_facts, gold, cause_names)
    return {"setup": name, "recall": round(valid.found.mean(), 3), "found": int(valid.found.sum()), "of": len(valid),
            "traps extracted": int(traps.extracted.sum())}, valid, traps

if HAVE_GOLD:
    gold = pd.read_csv(GOLD)
    empty = pd.DataFrame(columns=["relation", "subject", "object", "rank", "gas", "section", "subject_id", "object_id"])
    rows = [report("rules only (no LLM)", empty, {})[0]]
    if OLLAMA_OK:
        r, valid, traps = report("full pipeline", facts, cause_names)
        rows.append(r)
    display(pd.DataFrame(rows))


### Recall per fact type and per hard case
Where does extraction fail? The hard cases are the traps built into the manual: tables split by a paragraph,
facts only in running text, cross-references, figure captions, near-duplicate codes...


In [ ]:
if HAVE_GOLD and OLLAMA_OK:
    display(valid.groupby(["kind", "relation"])["found"].agg(["mean", "sum", "count"]).round(2))
    display(valid.fillna({"hard_case": "(none)"}).groupby("hard_case")["found"].agg(["mean", "sum", "count"]).round(2))
    display(traps[["subject", "relation", "object", "section", "extracted"]])
    print("Missed facts:")
    display(valid[~valid.found][["kind", "subject", "relation", "object", "section", "hard_case"]])


In [ ]:
if HAVE_GOLD and OLLAMA_OK:
    # LLM facts that match no gold fact: check these by hand (true extra fact, or made up?)
    _, _, align = score(facts, rule_facts, gold, cause_names)
    f = facts.assign(subj_gold=facts["subject_id"].map(align))
    unmatched = f[f.relation.isin(["CAUSES", "FIXED_BY", "INDICATES"]) & f.subj_gold.isna() & f.object_id.map(align).isna()]
    print(len(unmatched), "LLM facts about causes that match no gold cause:")
    display(unmatched[["relation", "subject", "object", "section", "evidence"]].head(20))


## 10. Ablation: what does each step add?
Each row turns one step off and scores again, reusing the cached LLM answers where possible:
- **no grounding check** and **no scope filter**: same LLM answers, one filter off;
- **no cross-references**: only the chunks that had a referenced section are sent again without it (a few calls).

This table is the evidence for the method chapter: it shows which step earns its place.


In [ ]:
RUN_NO_REFS = True
if HAVE_GOLD and OLLAMA_OK:
    rows = [report("full pipeline", facts, cause_names)[0]]
    for name, kw in [("no grounding check", {"use_grounding": False}), ("no scope filter", {"use_scope": False})]:
        f2, n2 = keep_causes(*canonicalise(check(raw, **kw)[0]))
        rows.append(report(name, f2, n2)[0])
    f2, n2 = canonicalise(check(raw)[0])
    rows.append({**report("no cause filter", f2, n2)[0], "cause names": len(n2)})
    rows[0]["cause names"] = len(cause_names)
    if RUN_NO_REFS:
        with_refs_ids = {c["chunk_id"] for c in chunks if c["refs"]}
        p = extract(use_refs=False, tag="norefs", only=with_refs_ids)
        raw_nr = pd.concat([raw[~raw.chunk_id.isin(with_refs_ids)], load_raw(p)], ignore_index=True)
        # the no-refs answers were made without the appended text, so ground them on the chunk alone
        by_id = {c["chunk_id"]: c["text"] for c in chunks}
        raw_nr.loc[raw_nr.chunk_id.isin(with_refs_ids), "chunk_text"] = raw_nr["chunk_id"].map(by_id)
        rule_nr = rule_facts.copy()
        rule_nr.loc[rule_nr.relation == "ALARM_ACTION", "object"] = rule_nr["object"].str.replace(r" \[.*", "", regex=True)
        f2, n2 = keep_causes(*canonicalise(check(raw_nr)[0]))
        v2, t2, _ = score(f2, rule_nr, gold, n2)
        rows.append({"setup": "no cross-references", "recall": round(v2.found.mean(), 3), "found": int(v2.found.sum()),
                     "of": len(v2), "traps extracted": int(t2.extracted.sum())})
    display(pd.DataFrame(rows))


## What to write in the report
- The pipeline turns a 66-page manual into graph facts with a **section and page for every fact** (provenance), the same
  property notebook 04's `source` gives each node.
- Report **recall overall, per fact type and per hard case**, the **traps extracted**, and the **ablation** table.
- Honest limits: the cutting-table and message-list parsers are written for this manual's table layout; the gold list
  covers diagnostic facts only, so precision needs a hand check; an 8B model's results vary with the model and prompt.
- With a real manual: run steps 1-4 first and look at the sections and tables it finds, then adjust the heading rule and
  the two table parsers, then run step 5. Nothing else depends on this manual.
